In [1]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 3  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [2]:
TEAM = "3"
budget_A = 1000000
budget_B = 750000
buy_date = pd.Timestamp("2026-11-27")
start = "2025-10-01"
end = "2026-09-30"
fx_symbol = "USDCAD=X"

# The TA file takes priority. The example has no header and includes a duplicate.
try:
    ticker_file = "Tickers.csv"
    ticker_column = pd.read_csv(ticker_file, header=None, dtype=str)[0]
except FileNotFoundError:
    ticker_file = "Tickers_Example.csv"
    ticker_column = pd.read_csv(ticker_file, header=None, dtype=str)[0]

tickers = ticker_column.dropna().str.strip().str.upper().drop_duplicates()
tickers = tickers[~tickers.isin(["", "TICKER", "TICKERS", "SYMBOL"])].tolist()
print("Input file:", ticker_file, "| Unique tickers:", len(tickers))

# Use the team's quote type, listing market, currency and sector checks.
rows = []
for ticker in tickers:
    try:
        info = yf.Ticker(ticker).get_info()
    except Exception as error:
        print(ticker, "metadata unavailable:", type(error).__name__)
        info = {}
    rows.append([ticker, info.get("longName"), info.get("quoteType"),
                 info.get("market"), info.get("currency"), info.get("sector"), info.get("beta")])

all_stocks = pd.DataFrame(rows, columns=["Ticker", "Company", "Type", "Market",
                                         "Currency", "Sector", "Yahoo_Beta"]).set_index("Ticker")
mask = (all_stocks["Type"].eq("EQUITY")
        & all_stocks["Market"].isin(["us_market", "ca_market"])
        & all_stocks["Currency"].isin(["CAD", "USD"])
        & all_stocks["Sector"].fillna("").str.strip().ne(""))
eligible_stocks = all_stocks.loc[mask].copy()
if eligible_stocks.empty:
    raise ValueError("No usable company profiles. Check the input file and Yahoo Finance.")
display(all_stocks.assign(Profile_pass=mask))

Input file: Tickers_Example.csv | Unique tickers: 40


,Company,Type,Market,Currency,Sector,Yahoo_Beta,Profile_pass
Ticker,,,,,,,
AAPL,Apple Inc.,EQUITY,us_market,USD,Technology,1.089,True
ABBV,AbbVie Inc.,EQUITY,us_market,USD,Healthcare,0.192,True
ABT,Abbott Laboratories,EQUITY,us_market,USD,Healthcare,0.537,True
ACN,Accenture plc,EQUITY,us_market,USD,Technology,1.004,True
AGN,NaN,NONE,NaN,NaN,NaN,NaN,False
AIG,"American International Group, Inc.",EQUITY,us_market,USD,Financial Services,0.475,True
AMZN,"Amazon.com, Inc.",EQUITY,us_market,USD,Consumer Cyclical,1.503,True
AXP,American Express Company,EQUITY,us_market,USD,Financial Services,1.077,True
BA,The Boeing Company,EQUITY,us_market,USD,Industrials,1.331,True


In [3]:
# Yahoo's end date is exclusive. A single bulk request also supplies historical FX.
today_et = pd.Timestamp.now(tz="America/Toronto")
official = today_et >= pd.Timestamp("2026-11-27 16:00", tz="America/Toronto")
price_date = buy_date if official else pd.Timestamp(end)
print("Official purchase date:" if official else "Practice prices only:", price_date.date())
benchmark = "^GSPC"
one_year_ago = price_date - pd.DateOffset(years=1)
# Extra trading days provide a previous close if the anniversary was a holiday.
download_start = min(pd.Timestamp(start), one_year_ago - pd.Timedelta(days=7))
symbols = eligible_stocks.index.tolist() + [fx_symbol, benchmark]
history = yf.download(symbols, start=download_start, end=price_date + pd.Timedelta(days=1),
                      group_by="ticker", auto_adjust=False, progress=False, threads=4)
def get_field(field):
    values = history.xs(field, level=1, axis=1).reindex(columns=symbols)
    values.index = pd.to_datetime(values.index).tz_localize(None).normalize()
    return values.sort_index()

close = get_field("Close")
volume = get_field("Volume")
if price_date not in close.index or pd.isna(close.at[price_date, fx_symbol]):
    raise ValueError("FX close unavailable for the pricing date. Retry when Yahoo publishes it.")
rate = float(close.at[price_date, fx_symbol])
cad_prices = {}
for ticker in eligible_stocks.index:
    traded = volume.loc[start:end, ticker].where(close.loc[start:end, ticker].notna()).dropna()
    counts = traded.groupby(traded.index.to_period("M")).transform("count")
    eligible_stocks.loc[ticker, "Avg_Volume"] = traded[counts >= 18].mean()
    eligible_stocks.loc[ticker, "Close"] = close.at[price_date, ticker]
    prices = close.loc[start:end, ticker]
    factor = 1.0
    if eligible_stocks.at[ticker, "Currency"] == "USD":
        prices = prices.ffill() * close.loc[start:end, fx_symbol].ffill()
        factor = rate
    cad_prices[ticker] = prices
    eligible_stocks.loc[ticker, "Close_CAD"] = close.at[price_date, ticker] * factor

eligible_stocks = eligible_stocks.loc[(eligible_stocks["Avg_Volume"] >= 5000)
                                      & (eligible_stocks["Close_CAD"] > 0)].copy()
# Fixed shares, price returns only. Carry the last close across market holidays.
# Score every candidate on the same dates, starting after all have prices.
stock_days = close.loc[start:end, eligible_stocks.index].notna().any(axis=1)
comparison_prices = pd.DataFrame(cad_prices)[eligible_stocks.index].loc[stock_days].ffill().dropna()
prices_cad = comparison_prices.copy()
daily_returns = prices_cad.pct_change().dropna()
vol = daily_returns.std()
usable = vol[(vol > 0) & np.isfinite(vol)].index
prices_cad = prices_cad[usable]
daily_returns = daily_returns[usable]
vol = vol[usable]
if len(usable) < 10 or len(daily_returns) < 2:
    raise ValueError("Too few usable stocks or common return dates. Inspect the Yahoo data.")
display(eligible_stocks)
print("Usable stocks:", len(usable), "| Common daily returns:", len(daily_returns))

def calc_fee_cad(shares_num):
    return min(0.001 * shares_num, 2.15) * rate

Practice prices only: 2026-09-30


,Company,Type,Market,Currency,Sector,Yahoo_Beta,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,,,
AAPL,Apple Inc.,EQUITY,us_market,USD,Technology,1.089,4.828203e+07,333.019989,472.598646
ABBV,AbbVie Inc.,EQUITY,us_market,USD,Healthcare,0.192,6.281745e+06,261.589996,371.230203
ABT,Abbott Laboratories,EQUITY,us_market,USD,Healthcare,0.537,1.008423e+07,98.830002,140.252617
ACN,Accenture plc,EQUITY,us_market,USD,Technology,1.004,6.116120e+06,183.369995,260.225855
AIG,"American International Group, Inc.",EQUITY,us_market,USD,Financial Services,0.475,4.324195e+06,74.419998,105.611650
AMZN,"Amazon.com, Inc.",EQUITY,us_market,USD,Consumer Cyclical,1.503,4.652561e+07,249.149994,353.576223
AXP,American Express Company,EQUITY,us_market,USD,Financial Services,1.077,3.054224e+06,304.100006,431.557432
BA,The Boeing Company,EQUITY,us_market,USD,Industrials,1.331,7.140954e+06,186.050003,264.029135
BAC,Bank of America Corporation,EQUITY,us_market,USD,Financial Services,1.192,3.673368e+07,54.430000,77.243245


Usable stocks: 35 | Common daily returns: 255


In [4]:
def capped_weights(scores, caps, total):
    caps = np.array(caps)
    result = np.zeros(len(scores))
    free = caps > 0
    remaining = total
    while free.any() and remaining > 1e-12:
        # Only update positions that are still open.
        proposal = result.copy()
        proposal[free] = scores[free] / scores[free].sum() * remaining
        over = free & (proposal > caps)
        if not over.any():
            return proposal
        # Fix the positions above their caps before the next round.
        result[over] = caps[over]
        free[over] = False
        remaining = total - result.sum()
    return result

## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [5]:
# Score Client A on a separate table so its filters do not change Client B.
eligible_A = eligible_stocks.copy()
fx_daily = close[fx_symbol].ffill()
market_close_cad = close[benchmark] * fx_daily
eligible_A["Return %"] = np.nan
for ticker in eligible_A.index:
    quoted = close[ticker].dropna()
    stock_close = quoted.loc[one_year_ago:price_date]
    if eligible_A.at[ticker, "Currency"] == "USD":
        stock_close = stock_close * fx_daily.reindex(stock_close.index)

    # Estimate beta over the trailing year in CAD for both stock and benchmark.
    both = pd.concat([stock_close.rename("stock"), market_close_cad.rename("market")],
                     axis=1).loc[one_year_ago:price_date].dropna()
    both_returns = both.pct_change().dropna()
    market_variance = both_returns["market"].var()
    if len(both_returns) > 1 and market_variance > 0:
        beta = both_returns["stock"].cov(both_returns["market"]) / market_variance
    else:
        beta = eligible_A.at[ticker, "Yahoo_Beta"]
        if pd.isna(beta):
            beta = 1.0
    eligible_A.at[ticker, "Beta"] = beta

    # Price momentum uses the stock's quoted currency, as in the growth strategy.
    old_price = quoted.asof(one_year_ago)
    if pd.notna(old_price) and old_price > 0:
        eligible_A.loc[ticker, "Return %"] = (quoted.loc[price_date] / old_price - 1) * 100

eligible_A = eligible_A.dropna(subset=["Beta", "Return %"]).copy()
if len(eligible_A) < 10:
    raise ValueError("Client A needs at least ten eligible stocks with a one-year return.")

def scaled_score(series):
    spread = series.max() - series.min()
    if spread == 0:
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / spread

eligible_A["Score"] = (scaled_score(eligible_A["Beta"]) + scaled_score(eligible_A["Return %"])) / 2

C:\Users\1\AppData\Local\Temp\ipykernel_51348\113455357.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  both = pd.concat([stock_close.rename("stock"), market_close_cad.rename("market")],
C:\Users\1\AppData\Local\Temp\ipykernel_51348\113455357.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  both = pd.concat([stock_close.rename("stock"), market_close_cad.rename("market")],
C:\Users\1\AppData\Local\Temp\ipykernel_51348\113455357.py:13: Pandas4Warning: Sorting by default when concatena

C:\Users\1\AppData\Local\Temp\ipykernel_51348\113455357.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  both = pd.concat([stock_close.rename("stock"), market_close_cad.rename("market")],
C:\Users\1\AppData\Local\Temp\ipykernel_51348\113455357.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  both = pd.concat([stock_close.rename("stock"), market_close_cad.rename("market")],
C:\Users\1\AppData\Local\Temp\ipykernel_51348\113455357.py:13: Pandas4Warning: Sorting by default when concatena

In [6]:
#select sectors the portfolio is going to focus on
growth_sectors = ['Technology', 'Communication Services']

mask = eligible_A['Sector'].isin(growth_sectors)
growth_stocks = eligible_A[mask].sort_values('Score', ascending=False)
other_stocks = eligible_A[~mask].sort_values('Score', ascending=False)

# total stock count
n_total = 15
# use 5 selected sector stocks, or fewer if the ticker file doesn't have 5
n_growth = min(5, len(growth_stocks))   
n_other = n_total - n_growth

#take the n_growth number of the best stocks in the selected sectors
selected_growth = growth_stocks.head(n_growth).copy()
selected_other = other_stocks.head(n_other).copy()

# If a bucket has too few names or too little sector capacity, fill by score.
# A stock adds room until its sector reaches the 40% limit.
def basket_capacity(tickers_A):
    counts = eligible_A.loc[tickers_A, "Sector"].value_counts()
    return (counts * 0.15).clip(upper=0.40).sum()

chosen_A = selected_growth.index.tolist() + selected_other.index.tolist()
target_count = min(n_total, len(eligible_A))
preferred_A = chosen_A.copy()
ranked_A = eligible_A.sort_values("Score", ascending=False, kind="stable").index.tolist()
if len(chosen_A) < target_count or basket_capacity(chosen_A) < 1 - 1e-10:
    chosen_A = []
    order_A = preferred_A + [ticker for ticker in ranked_A if ticker not in preferred_A]
    while len(chosen_A) < target_count:
        remaining_A = [ticker for ticker in order_A if ticker not in chosen_A]
        chosen_A.append(max(remaining_A, key=lambda ticker: min(1.0, basket_capacity(chosen_A + [ticker]))))
    if basket_capacity(chosen_A) < 1 - 1e-10:
        raise ValueError("Client A cannot invest fully within the sector limits.")
    chosen_table = eligible_A.loc[chosen_A]
    in_growth = chosen_table["Sector"].isin(growth_sectors)
    selected_growth = chosen_table[in_growth].copy()
    selected_other = chosen_table[~in_growth].copy()
    n_growth = len(selected_growth)

In [7]:
tolerance = 0.0000001

def cap_weights(df, cap=0.15):
    while (df['Weight'] > cap + tolerance).any():
        excess = 0
        for ticker in df.index:
            if df.loc[ticker, 'Weight'] > cap:
                excess = excess + (df.loc[ticker, 'Weight'] - cap)
                df.loc[ticker, 'Weight'] = cap
    
        not_capped = df[df['Weight'] < cap]
        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

    return df

#for the 40% sector cap
def cap_sectors(df, cap=0.40):
    tries = 0
    while (df.groupby('Sector')['Weight'].sum() > cap + tolerance).any() and tries < 50:
        tries = tries + 1
        sector_weights = df.groupby('Sector')['Weight'].sum()
        excess = 0

        for sec in sector_weights.index:
            if sector_weights[sec] > cap:
                excess = excess + (sector_weights[sec] - cap)
                mask = df['Sector'] == sec
                df.loc[mask, 'Weight'] = df.loc[mask, 'Weight'] * (cap / sector_weights[sec])

        under_sectors = sector_weights[sector_weights < cap - tolerance].index
        not_capped = df[df['Sector'].isin(under_sectors) & (df['Weight'] < 0.15 - tolerance)]
        if len(not_capped) == 0:
            break

        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

        df = cap_weights(df)

    return df

# can't ask for more than the 15% cap allows with this many stocks
target_growth = min(0.40, n_growth * 0.15)
target_other = 1 - target_growth

# A tiny score floor keeps a selected stock with the lowest score invested.
for bucket, target in [(selected_growth, target_growth), (selected_other, target_other)]:
    if not bucket.empty:
        score = bucket["Score"].clip(lower=1e-8)
        bucket["Weight"] = score / score.sum() * target
    else:
        bucket["Weight"] = pd.Series(dtype=float)

selected_growth = cap_weights(selected_growth)
selected_other = cap_weights(selected_other)

portfolio_A = pd.concat([selected_growth, selected_other])

starting_scores_A = portfolio_A["Weight"].copy()
portfolio_A = cap_sectors(portfolio_A)

# If an entire bucket was capped, allocate the unused money across feasible sectors.
sector_totals_A = portfolio_A.groupby("Sector")["Weight"].sum()
if (abs(portfolio_A["Weight"].sum() - 1) > 1e-10 or
        portfolio_A["Weight"].max() > 0.15 + 1e-10 or
        sector_totals_A.max() > 0.40 + 1e-10 or
        (portfolio_A["Weight"] <= 0).any()):
    scores_A = starting_scores_A.clip(lower=1e-8)
    sector_groups_A = portfolio_A.groupby("Sector", sort=False).groups
    sector_scores_A = np.array([scores_A.loc[group].sum() for group in sector_groups_A.values()])
    sector_caps_A = [min(0.40, 0.15 * len(group)) for group in sector_groups_A.values()]
    sector_amounts_A = capped_weights(sector_scores_A, sector_caps_A, 1.0)
    for group, amount in zip(sector_groups_A.values(), sector_amounts_A):
        portfolio_A.loc[group, "Weight"] = capped_weights(
            scores_A.loc[group].to_numpy(), [0.15] * len(group), amount)

display(portfolio_A[["Sector", "Beta", "Return %", "Score", "Weight"]]
        .sort_values("Weight", ascending=False))

,Sector,Beta,Return %,Score,Weight
Ticker,,,,,
BB.TO,Technology,1.463962,89.101623,0.861114,0.102149
CAT,Industrials,1.710099,69.923502,0.839565,0.090054
TXN,Technology,1.474732,52.446526,0.727111,0.086253
QCOM,Technology,2.151791,10.627550,0.708460,0.084040
SHOP.TO,Technology,1.665039,2.011996,0.578167,0.068584
C,Financial Services,1.314110,27.566498,0.602246,0.064598
MRK,Healthcare,0.347486,73.132369,0.576349,0.061821
BIIB,Healthcare,0.495532,61.250715,0.562100,0.060292
AMZN,Consumer Cyclical,1.359937,13.471779,0.559136,0.059974


### Client A — Financial justification

### Why our Portfolio A strategy makes financial sense

We designed Portfolio A for a client willing to accept more risk in pursuit of higher growth. We first remove stocks that do not meet the assignment’s eligibility and other rules that were mentioned. For each remaining stock, we estimate beta from daily CAD returns over the trailing year, comparing the stock with the S&P 500 valued in CAD. Beta measures how sensitive a stock’s past returns have been to market movements. For comparison a beta above 1 has historically moved more than the market, while a beta below 1 has moved less. We combine beta with the previous year’s price return, giving both measures equal importance after scaling them. This favours stocks that have recently performed well and have shown greater market sensitivity, which fits a higher risk objective, although neither measure guarantees future returns.

We selected up to five of the highest scoring eligible stocks in Technology and Communication Services for a growth allocation of up to 40%. We select the rest from other sectors. Within those groups, higher scores receive larger starting weights, subject to a 15% limit for any one stock and a 40% limit for any one sector. These limits reduce the chance that one company or sector dominates the portfolio while keeping the growth oriented allocation. We convert USD holdings to CAD for portfolio values and fees  and are using fractional shares to invest close to the budget.

### What the evidence and equal-weight comparison show

In the course example priced on September 30, 2026, both portfolios held the same 15 stocks. From October 1, 2025 to September 30, 2026, the growth weights returned 42.924%, compared with 39.733% at equal weights. Daily CAD volatility also rose from 1.110% to 1.238%. The growth client earned a higher historical return in exchange for larger daily swings. These are cumulative price returns with fixed shares. Betas and past returns helped choose the stocks, so this sample also reflects how the stocks were selected. The comparison table updates when the inputs change.

### What could go wrong

Our method can perform poorly if the market turns against higher-beta or recently strong stocks. Beta and past returns describe the period used to calculate them; neither guarantees future performance, and recent winners can reverse. Stocks can also become more correlated during a market decline, so holding several companies does not remove broad market risk. The sector and individual-stock limits reduce concentration, but they do not prevent losses, and the growth allocation can still make the portfolio sensitive to changes in technology valuations.

Because we hold Canadian and U.S. stocks, changes in the USD/CAD exchange rate can raise or lower the CAD value of U.S. holdings even when their USD prices do not change. Our historical comparison covers a limited period and ignores dividends as required by the assignment, so it may not represent other market conditions or total investor experience.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

In [8]:
# Columns are each stock's CAD value after investing one dollar on the first day.
growth = (prices_cad / prices_cad.iloc[0]).values

# Keep sectors and volatility in the same order as the price columns.
names = list(prices_cad.columns)
sectors = list(eligible_stocks.loc[names, "Sector"])
stock_vol = vol.loc[names].values

def path_stats(value):#take out the comparable days.
    #the day to the end and the last second day. so that we can calculate the return of that period
    returns = value[1:] / value[:-1] - 1#we will get a list of returns here.
    total_return = value[-1] / value[0] - 1
    daily_volatility = returns.std(ddof=1)  # Sample standard deviation
    return total_return, daily_volatility

def sector_groups(ids):#divide them into different industries
    groups = {}
    for pos, stock in enumerate(ids):
        sector = sectors[stock]
        if sector not in groups:
            groups[sector] = []
        groups[sector].append(pos)  # Position in the selected portfolio
    return groups

def sector_capacity(groups):
    capacity = 0.0
    for group in groups.values():
        # Each stock is capped at 15% and each sector at 40%.
        sector_max = min(0.15 * len(group), 0.40)
        capacity += sector_max
    return capacity

def feasible_weights(ids, scores):
    groups = list(sector_groups(ids).values())
    sector_caps = [min(0.40, 0.15 * len(group)) for group in groups]
    if sum(sector_caps) < 1 - 1e-10:
        return None
    sector_scores = np.array([scores[group].sum() for group in groups])
    # Allocate across sectors first, then split each sector's weight among its stocks.
    sector_weights = capped_weights(sector_scores, sector_caps, 1.0)
    weights = np.zeros(len(ids))
    for group, sector_weight in zip(groups, sector_weights):
        weights[group] = capped_weights(scores[group], [0.15] * len(group), sector_weight)
    return weights

def score_final(ids):
    best = None#trying two different methods.
    for label, scores in [
        ("Equal before caps", np.ones(len(ids))),
        ("Inverse volatility before caps", 1 / stock_vol[ids])
    ]:
        weights = feasible_weights(ids, scores)
        if weights is None:
            return None
        portfolio_value = (growth[:, ids] * weights).sum(axis=1)
        risk = path_stats(portfolio_value)[1]
        if best is None or risk < best[0]:
            best = (risk, ids.copy(), weights, label)
    return best

In [9]:
best = None
stock_risks = [(stock_vol[i], i) for i in range(len(names))]
# Try the three lowest-volatility stocks as starting points.
for volatility, seed in sorted(stock_risks)[:3]:
    ids = [seed]
    while len(ids) < min(20, len(names)):
        current = growth[:, ids].sum(axis=1)
        candidates = []
        for stock in range(len(names)):
            if stock in ids:
                continue
            trial = ids + [stock]
            capacity = sector_capacity(sector_groups(trial))
            # Treat baskets that can reach 100% as tied on capacity.
            capped_capacity = round(min(capacity, 1.0), 8)
            # Compare daily volatility with equal starting weights.
            risk = path_stats(current + growth[:, stock])[1]
            candidates.append((-capped_capacity, risk, names[stock], stock))
        ids.append(min(candidates)[-1])
        if len(ids) >= 10:
            result = score_final(ids)
            if result is not None and (best is None or result[0] < best[0]):
                best = result

if best is None:
    raise ValueError("No feasible Client B portfolio. Review ticker coverage and sectors.")

# Test each one-stock swap using the same starting basket.
starting_ids = best[1].copy()
for pos in range(len(starting_ids)):
    for stock in range(len(names)):
        if stock in starting_ids:
            continue
        trial = starting_ids.copy()
        trial[pos] = stock
        result = score_final(trial)
        if result is not None and result[0] < best[0]:
            best = result

risk, selected_ids, weights, weighting_rule = best
selected = [names[i] for i in selected_ids]
correlations = daily_returns[selected].corr()
pair_correlations = []
for i in range(len(selected)):
    # Start at the next stock so each pair is counted once.
    for j in range(i + 1, len(selected)):
        pair_correlations.append(correlations.iloc[i, j])
mean_correlation = sum(pair_correlations) / len(pair_correlations)
print("Weighting rule:", weighting_rule)
print("Mean pairwise CAD return correlation:", round(mean_correlation, 3))
display(correlations.round(2))

Weighting rule: Inverse volatility before caps
Mean pairwise CAD return correlation: 0.073


,TD.TO,KO,T.TO,RY.TO,LMT,MO,AMZN,ABBV,AIG,TXN
TD.TO,1.00,-0.06,0.01,0.81,0.08,-0.13,0.24,0.00,0.17,0.22
KO,-0.06,1.00,0.16,-0.06,0.07,0.45,-0.06,0.37,0.23,-0.04
T.TO,0.01,0.16,1.00,-0.02,-0.10,0.10,-0.21,0.05,-0.01,-0.04
RY.TO,0.81,-0.06,-0.02,1.00,0.08,-0.09,0.21,0.02,0.13,0.16
LMT,0.08,0.07,-0.10,0.08,1.00,0.05,-0.04,0.13,0.04,-0.06
MO,-0.13,0.45,0.10,-0.09,0.05,1.00,-0.10,0.29,0.26,-0.10
AMZN,0.24,-0.06,-0.21,0.21,-0.04,-0.10,1.00,-0.17,-0.01,0.14
ABBV,0.00,0.37,0.05,0.02,0.13,0.29,-0.17,1.00,0.15,-0.02
AIG,0.17,0.23,-0.01,0.13,0.04,0.26,-0.01,0.15,1.00,-0.05
TXN,0.22,-0.04,-0.04,0.16,-0.06,-0.10,0.14,-0.02,-0.05,1.00


### Client B — Holy Grail diversification

**Why these stocks?** Our client wants a steady portfolio in Canadian dollars. We use Ray Dalio’s Holy Grail idea: hold stocks that do not all move together. We try three starting stocks with low daily volatility. At each step we test every remaining stock with the current basket on the same dates. We give priority to having enough room under the sector and stock limits. We then choose the addition with the lowest combined volatility at equal initial weights. This measure reflects both the stock’s own risk and how it moves with our holdings.

**How much do we invest?** For each feasible basket of 10–20 stocks, we test equal starting amounts and inverse-volatility weights. The second rule gives less money to more volatile stocks. We cap sectors and individual positions, keep the lowest historical volatility we find and test one round of single-stock replacements. This is a simple search, so another combination could do better.

**What evidence do we use?** In the course example priced on September 30, 2026, we used fixed starting shares. From October 1, 2025 to September 30, 2026, the average
correlation between pairs of stocks was 0.073. Both portfolios held the same 10 stocks
and kept the number of shares fixed. Our starting weights gave Client B lower daily
CAD volatility (0.609%) than starting with 10% in each stock (0.653%). Our portfolio
also returned 17.015% over the period, compared with 15.642% for equal weights.

**What could go wrong?** We choose and compare weights on the same historical data. This can make the result look better than it will be next week. Even stocks from different sectors may fall together. USD holdings also expose the client to changes in the exchange rate.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [10]:
# Same dates for both clients and their equal-weight comparisons.
# Fixed starting shares make weights drift with prices.
def compare_portfolios(client, tickers, allocation, strategy):
    prices = comparison_prices.loc[:, tickers]
    relative_prices = prices / prices.iloc[0]
    rows = []
    for label, initial_weights in [(strategy, np.array(allocation)),
                                   ("Same stocks, equal weight", np.ones(len(tickers)) / len(tickers))]:
        value = (relative_prices.to_numpy() * initial_weights).sum(axis=1)
        total_return, daily_std = path_stats(value)
        rows.append([label, 100 * total_return, 100 * daily_std, 100 * daily_std * np.sqrt(252)])
    table = pd.DataFrame(rows, columns=["Portfolio", "Historical return (%)", "Daily CAD volatility (%)",
                                        "Annualized volatility (%)"]).set_index("Portfolio")
    print(f"Client {client} | Fixed shares from {prices.index[0].date()} to {prices.index[-1].date()}")
    display(table.round(3))
    difference = table.iloc[0] - table.iloc[1]
    print(f"Strategy minus equal weight: {difference.iloc[0]:+.3f} percentage points of return, "
          f"{difference.iloc[1]:+.3f} points of daily volatility.")
    return table

comparison_A = compare_portfolios("A", portfolio_A.index.tolist(), portfolio_A["Weight"], "Growth scores")
comparison_B = compare_portfolios("B", selected, weights, "Holy Grail")

Client A | Fixed shares from 2025-10-01 to 2026-09-30


,Historical return (%),Daily CAD volatility (%),Annualized volatility (%)
Portfolio,,,
Growth scores,42.924,1.238,19.652
"Same stocks, equal weight",39.733,1.110,17.625


Strategy minus equal weight: +3.191 percentage points of return, +0.128 points of daily volatility.
Client B | Fixed shares from 2025-10-01 to 2026-09-30


,Historical return (%),Daily CAD volatility (%),Annualized volatility (%)
Portfolio,,,
Holy Grail,17.015,0.609,9.671
"Same stocks, equal weight",15.642,0.653,10.373


Strategy minus equal weight: +1.373 percentage points of return, -0.044 points of daily volatility.


## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [11]:
# Solve for the largest investment that fits each client's budget after per-stock fees.
def build_holdings(tickers, weights, budget):
    prices = eligible_stocks.loc[tickers, "Close_CAD"].to_numpy()
    def cost(investment):
        shares = investment * weights / prices
        return float(np.dot(shares, prices)) + sum(calc_fee_cad(q) for q in shares)

    low, high = 0.0, budget  # Try to spend all the budget.
    for step in range(55):
        middle = (low + high) / 2
        if cost(middle) <= budget - 0.000001:
            low = middle
        else:
            high = middle
    shares = low * weights / prices
    values = shares * prices
    return pd.DataFrame({
        "Ticker": tickers, "Price": eligible_stocks.loc[tickers, "Close"].to_numpy(),
        "Currency": eligible_stocks.loc[tickers, "Currency"].to_numpy(),
        "Shares": shares, "Value": values, "Weight": values / values.sum()
    }, index=range(1, len(tickers) + 1))

Portfolio_Final_A = build_holdings(portfolio_A.index.tolist(), portfolio_A["Weight"].to_numpy(), budget_A)
Portfolio_Final_B = build_holdings(selected, weights, budget_B)

In [12]:
def validate_portfolio(portfolio, budget):
    tickers = portfolio["Ticker"]
    eligible = eligible_stocks.reindex(tickers)
    sectors = tickers.map(eligible_stocks["Sector"])
    sector_weights = portfolio.groupby(sectors)["Weight"].sum()
    fees = sum(calc_fee_cad(q) for q in portfolio["Shares"])
    spent = portfolio["Value"].sum() + fees
    return {
        "Tickers are eligible and unique": tickers.is_unique and tickers.isin(eligible_stocks.index).all()
            and eligible["Type"].eq("EQUITY").all() and eligible["Currency"].isin(["CAD", "USD"]).all()
            and eligible["Market"].isin(["ca_market", "us_market"]).all() and sectors.notna().all()
            and (eligible["Avg_Volume"] >= 5000).all(),
        "10 to 20 stocks": 10 <= len(portfolio) <= 20,
        "Long only, at most 15% per stock": (portfolio["Weight"] > 0).all()
            and portfolio["Weight"].max() <= 0.15 + 1e-10,
        "At most 40% per sector": sector_weights.max() <= 0.40 + 1e-10,
        "Weights sum to 100%": np.isclose(portfolio["Weight"].sum(), 1),
        "Purchases plus fees fit the budget": spent <= budget and budget - spent < 0.01
    }

checks = pd.DataFrame({"Client A": validate_portfolio(Portfolio_Final_A, budget_A),
                       "Client B": validate_portfolio(Portfolio_Final_B, budget_B)})
display(checks)
assert checks.all().all(), "Review failed checks before exporting."

sector_exposure = {}
for client, portfolio in [("A", Portfolio_Final_A), ("B", Portfolio_Final_B)]:
    sectors_held = portfolio["Ticker"].map(eligible_stocks["Sector"])
    sector_exposure[client] = portfolio.groupby(sectors_held)["Weight"].sum() * 100
print("Sector weights (%)")
display(pd.DataFrame(sector_exposure).fillna(0).round(2))

,Client A,Client B
Tickers are eligible and unique,True,True
10 to 20 stocks,True,True
"Long only, at most 15% per stock",True,True
At most 40% per sector,True,True
Weights sum to 100%,True,True
Purchases plus fees fit the budget,True,True


Sector weights (%)


,A,B
Ticker,,
Communication Services,0.00,10.09
Consumer Cyclical,6.00,6.88
Consumer Defensive,0.00,20.77
Financial Services,22.43,39.54
Healthcare,17.65,9.00
Industrials,13.92,8.22
Technology,40.00,5.50


## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [13]:
print("Official pricing" if official else "Practice pricing", "| Price date:", price_date.date())
for client, portfolio, client_budget in [("A", Portfolio_Final_A, budget_A), ("B", Portfolio_Final_B, budget_B)]:
    purchases = portfolio["Value"].sum()
    fees = sum(calc_fee_cad(q) for q in portfolio["Shares"])
    print(f"Client {client} | Budget: CAD {client_budget:,.2f} | Purchases: CAD {purchases:,.2f} | "
          f"Fees: CAD {fees:,.2f} | Cash: CAD {client_budget - purchases - fees:,.2f}")
    display(portfolio)
    portfolio[["Ticker", "Shares"]].to_csv(f"Stocks_Group_{TEAM}_{client}.csv", index=False)

Practice pricing | Price date: 2026-09-30
Client A | Budget: CAD 1,000,000.00 | Purchases: CAD 999,991.82 | Fees: CAD 8.18 | Cash: CAD 0.00


,Ticker,Price,Currency,Shares,Value,Weight
1,BB.TO,12.840000,CAD,7955.445908,102147.926677,0.102149
2,TXN,280.089996,USD,216.994981,86252.056352,0.086253
3,QCOM,184.039993,USD,321.772943,84039.585675,0.084040
4,SHOP.TO,210.919998,CAD,325.165473,68583.900980,0.068584
5,AAPL,333.019989,USD,124.785072,58973.256329,0.058974
6,CAT,810.789978,USD,78.265012,90053.002312,0.090054
7,C,129.479996,USD,351.554712,64597.806498,0.064598
8,MRK,145.309998,USD,299.786146,61820.031931,0.061821
9,BIIB,225.880005,USD,188.086507,60291.709871,0.060292
10,AMZN,249.149994,USD,169.620360,59973.726278,0.059974


Client B | Budget: CAD 750,000.00 | Purchases: CAD 749,991.69 | Fees: CAD 8.31 | Cash: CAD 0.00


,Ticker,Price,Currency,Shares,Value,Weight
1,TD.TO,167.330002,CAD,654.349849,109492.361455,0.145991
2,KO,86.080002,USD,735.786997,89882.790569,0.119845
3,T.TO,11.950000,CAD,6330.514462,75649.646615,0.100867
4,RY.TO,279.940002,CAD,401.867374,112498.753782,0.150000
5,LMT,509.250000,USD,85.271794,61625.237989,0.082168
6,MO,67.339996,USD,689.339863,65876.217225,0.087836
7,AMZN,249.149994,USD,146.034467,51634.315211,0.068847
8,ABBV,261.589996,USD,181.806664,67492.124734,0.089990
9,AIG,74.419998,USD,706.091958,74571.536467,0.099430
10,TXN,280.089996,USD,103.824800,41268.707835,0.055026


## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line).

AI assistance was used to develop parts of the code and explanations and to integrate and test the two portfolios.